# Credit Scoring and PD Calibration

**Purpose:** Show how the `finstack_quant.models.credit` bindings support both classic bankruptcy-score style models and PiT/TtC PD transformations.

**Prerequisites:** Basic credit ratios and familiarity with point-in-time versus through-the-cycle default probabilities.

**In this notebook:** We compute Altman, Ohlson, and Zmijewski scores, then connect them to macro-conditioned PD calibration using the Merton-Vasicek mapping.


## Concept

This notebook combines two adjacent workflows:

1. **Cross-sectional scoring** from a company's balance-sheet and income-statement ratios.
2. **PD calibration** that converts long-run TtC views into stressed or benign PiT views.

Together they cover both the qualitative screening question and the quantitative default-probability question.


In [ ]:
import math

import sys
sys.path.insert(0, "../..")

from _shared import banner
from finstack_quant.models.credit import pd as credit_pd
from finstack_quant.models.credit import scoring

# The scoring bindings take normalized, units-agnostic ratios, so the ratio prep
# below is call-site input assembly rather than modeling logic.
total_assets = 1_000.0

total_liabilities = 600.0
current_assets = 400.0
current_liabilities = 250.0
working_capital = current_assets - current_liabilities
retained_earnings = 200.0
ebit = 120.0
sales = 1_400.0
market_equity = 900.0
book_equity = total_assets - total_liabilities
net_income = 60.0
funds_from_operations = 140.0

wc_ta = working_capital / total_assets
re_ta = retained_earnings / total_assets
ebit_ta = ebit / total_assets
sales_ta = sales / total_assets
mkt_eq_liab = market_equity / total_liabilities
bk_eq_liab = book_equity / total_liabilities
tl_ta = total_liabilities / total_assets
cl_ca = current_liabilities / current_assets
ca_cl = current_assets / current_liabilities
ni_ta = net_income / total_assets
ffo_tl = funds_from_operations / total_liabilities


## Scores, zones, and macro conditioning

The next cell prints the sample company inputs, compares the standard score families, and then runs a PiT/TtC round-trip. Altman models provide canonical scores and zones but no native probability; calibrate score-to-PD mappings separately against a documented population and vintage.


In [ ]:
banner("Sample company financials")
print(f"Total assets         : {total_assets:,.0f}")
print(f"Total liabilities    : {total_liabilities:,.0f}")
print(f"Working capital      : {working_capital:,.0f}")
print(f"Retained earnings    : {retained_earnings:,.0f}")
print(f"EBIT                 : {ebit:,.0f}")
print(f"Sales                : {sales:,.0f}")
print(f"Market cap           : {market_equity:,.0f}")

banner("Altman / Ohlson / Zmijewski")
# Every scoring model returns a typed ScoringResult (score, zone, implied_pd, model).
base = {
    "working_capital_to_total_assets": wc_ta,
    "retained_earnings_to_total_assets": re_ta,
    "ebit_to_total_assets": ebit_ta,
}
z = scoring.altman_z_score(
    scoring.AltmanZScoreInput(**base, market_equity_to_total_liabilities=mkt_eq_liab, sales_to_total_assets=sales_ta)
)
zp = scoring.altman_z_prime(
    scoring.AltmanZPrimeInput(**base, book_equity_to_total_liabilities=bk_eq_liab, sales_to_total_assets=sales_ta)
)
zdp = scoring.altman_z_double_prime(
    scoring.AltmanZDoublePrimeInput(**base, book_equity_to_total_liabilities=bk_eq_liab)
)
o = scoring.ohlson_o_score(
    scoring.OhlsonOScoreInput(
        log_total_assets_adjusted=math.log(total_assets),
        total_liabilities_to_total_assets=tl_ta,
        working_capital_to_total_assets=wc_ta,
        current_liabilities_to_current_assets=cl_ca,
        liabilities_exceed_assets=1.0 if total_liabilities > total_assets else 0.0,
        net_income_to_total_assets=ni_ta,
        funds_from_operations_to_total_liabilities=ffo_tl,
        negative_net_income_two_years=0.0,
        net_income_change=0.05,
    )
)
zmijewski_ratios = {
    "net_income_to_total_assets": ni_ta,
    "total_liabilities_to_total_assets": tl_ta,
    "current_assets_to_current_liabilities": ca_cl,
}
y = scoring.zmijewski_score(scoring.ZmijewskiInput(**zmijewski_ratios))
z_zone, o_zone = z.zone, o.zone

print(f"Altman Z        : {z.score:.3f}  zone={z.zone:<8}")
print(f"Altman Z'       : {zp.score:.3f}  zone={zp.zone:<8}")
print(f"Altman Z''      : {zdp.score:.3f}  zone={zdp.zone:<8}")
print(f"Ohlson O-Score  : {o.score:.3f}  zone={o.zone:<8}  PD={o.implied_pd:.2%}")
print(f"Zmijewski Y     : {y.score:.3f}  zone={y.zone:<8}  PD={y.implied_pd:.2%}")

# Models with a native probability map straight onto a master scale.
master = credit_pd.MasterScale.default_calibration()
print(f"Ohlson grade    : {master.map_score(o).grade}")
print(f"Zmijewski grade : {master.map_score(y).grade}")

# credit_score_report scores and grades on the default master scale in one call.
report = scoring.credit_score_report({"model": "zmijewski_score", "input": zmijewski_ratios})
print(f"Report grade    : {report.grade.grade}")

banner("PiT / TtC conversion")
rho = 0.15
z_down = -1.5
z_bene = 1.0
ttc_pd = 0.020
pit_down = credit_pd.ttc_to_pit(ttc_pd, rho, z_down)
pit_bene = credit_pd.ttc_to_pit(ttc_pd, rho, z_bene)
rt_ttc = credit_pd.pit_to_ttc(pit_down, rho, z_down)
rt_pit = credit_pd.ttc_to_pit(rt_ttc, rho, z_down)

print(f"Input TtC PD        : {ttc_pd:.2%}")
print(f"Downturn PiT PD     : {pit_down:.2%}")
print(f"Benign PiT PD       : {pit_bene:.2%}")
print(f"Round-trip TtC      : {rt_ttc:.2%}")
print(f"Round-trip PiT      : {rt_pit:.2%}")

history = [0.015, 0.022, 0.018, 0.030, 0.012, 0.020, 0.025]
ct = credit_pd.central_tendency(history)
print(f"Central tendency    : {ct:.4%}")


## Scorecard extension note

This notebook demonstrates core `rating_scales` + PD term-structure usage. The
`statements_analytics` credit scorecard extension (`ScorecardMetric`/`Config`/`Report`,
`CreditScorecardExtension`) is a separate weighted custom-metric scorecard
builder. It runs on a scale from a registry the caller supplies; there is no
embedded or default scale, and threshold labels must be bands of that scale.

In [ ]:
from finstack_quant.core.rating_scales import (
    RatingLevel,
    RatingScaleEntry,
    RatingScaleRegistry,
    ScorecardScale,
)
from finstack_quant.statements_analytics import (
    ScorecardConfig,
    ScorecardMetric,
    ScorecardReport,
    CreditScorecardExtension,
)

# The caller owns the scale: its numbers are an internal utility policy.
scale = ScorecardScale(
    "Internal corporate",
    [RatingLevel("A", 90.0, 80.0), RatingLevel("B", 70.0, 50.0), RatingLevel("C", 20.0, 0.0)],
)
registry = RatingScaleRegistry([RatingScaleEntry("internal", "Example credit policy, 2026 review", scale)])

# A scorecard maps financial-ratio formulas + rating thresholds to an overall grade.
leverage = ScorecardMetric(
    name="leverage",
    formula="debt / ebitda",
    weight=0.6,
    thresholds={"A": (0.0, 2.0), "B": (2.0, 4.0), "C": (4.0, 99.0)},
)
coverage = ScorecardMetric(
    name="coverage",
    formula="ebitda / interest",
    weight=0.4,
    thresholds={"A": (6.0, 99.0), "B": (3.0, 6.0), "C": (0.0, 3.0)},
)
config = ScorecardConfig(rating_scale="internal", metrics=[leverage, coverage], period="2024Q4")
config.validate(registry)
print(f"ScorecardConfig: scale={config.rating_scale} metrics={[m.name for m in config.metrics]}")

# The extension snapshots the selected registry entry when it is built.
ext = CreditScorecardExtension(config, registry)
print("CreditScorecardExtension scale:", ext.rating_scale.id, "| source:", ext.rating_scale.source)
print("Produces a", ScorecardReport.__name__, "when executed against a model + results.")

## Takeaways

- The scoring functions give a quick, explainable view of financial distress risk.
- The `pd` helpers let you separate **structural credit quality** from **cycle state**.
- In real use, these are complementary: scorecards can inform screening, while PiT/TtC conversions can inform provisioning, stress tests, or credit portfolio scenarios.


In [ ]:
{
    "altman_zone": z_zone,
    "ohlson_zone": o_zone,
    "ttc_pd": round(ttc_pd, 4),
    "pit_down": round(pit_down, 4),
    "pit_bene": round(pit_bene, 4),
    "central_tendency": round(ct, 4),
}
